# **Roofline**

In [ ]:
import numpy as np
import plotly.graph_objects as go
import torch

from src.kompass.kernels.softmax import Softmax


In [2]:
gpu = "t4"
peak_mem_bandwidth = 320 # GB/S
peak_comp_througput_fp32 = 8.1 # TFLOPS

In [15]:
sm = Softmax()
bytes = np.array(
    [
        sm.bytes_moved(params={"N": 2**i, "M": 4096}, dtype=torch.float32)
        for i in range(30)
    ]
)
flops = np.array([sm.flops(params={"N": 2**i, "M": 4096}) for i in range(30)])
arithmetic_intensity = flops / bytes
x = np.linspace(0, 5, 30)

In [31]:
peak_mem_bandwidth = 320  # GB/s (Slope)
peak_comp_throughput = 8100  # GFLOPS (8.1 TFLOPS)

# x-axis representing Arithmetic Intensity (FLOPs/Byte)
x_roofline = np.linspace(0, 50, 500)

# The Roofline formula: min(Memory Bound, Compute Bound)
y_roofline = np.minimum(peak_mem_bandwidth * x_roofline, peak_comp_throughput)

# 2. Softmax characteristics
softmax_ai = 0.375  # 3 FLOPs / 8 Bytes
# Maximum possible performance Softmax can achieve on this hardware
softmax_max_perf = softmax_ai * peak_mem_bandwidth  # 120 GFLOPS
fig = go.Figure()
fig.add_traces(
    [
        go.Scatter(
            x=x_roofline,
            y=y_roofline,
            name="Hardware Roofline Ceiling",
            mode="lines",
        ),
        go.Scatter(
            x=[softmax_ai],
            y=[softmax_max_perf],
            name="Softmax Peak Potential",
            mode="markers",
            marker={"size": 12, "symbol": "circle"},
        ),
    ]
)

fig.update_layout(
    title="T4 Roofline Model vs. Softmax",
    xaxis_title="Arithmetic Intensity (FLOPs/Byte)",
    yaxis_title="Performance (GFLOPS)",
    template="plotly_white",
)

fig.show()